# Engram + Attention Residuals: Training a Small Language Model

This notebook demonstrates training a small Transformer model that combines:
- **Engram** (DeepSeek) - O(1) hash-based memory lookup for factual knowledge
- **Attention Residuals** (Kimi/Moonshot AI) - Learned attention over depth instead of fixed residual connections

The model is sized to train on a laptop CPU/GPU (< 10M parameters).

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import time
import os

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Import our modules
from engram import EngramModule
from attention_residuals import BlockAttnRes, RMSNorm
from combined_model import CombinedEngramAttnResTransformer

## 2. Configuration

These hyperparameters are tuned for laptop training (~5-10M params).

In [ ]:
# Model config (small enough for laptop)
CONFIG = {
    "vocab_size": 256,           # Character-level for simplicity
    "hidden_dim": 128,           # Small hidden dim
    "num_heads": 4,              # 4 attention heads
    "num_layers": 6,             # 6 transformer layers
    "ffn_dim": 256,              # FFN intermediate dim
    "block_size": 4,             # AttnRes block size (2 layers per block)
    "max_seq_len": 128,          # Context window
    "dropout": 0.1,
}

# Engram config
ENGRAM_CONFIG = {
    "vocab_size": 256,
    "compressed_vocab_size": 200,
    "engram_dim": 16,
    "num_heads": 4,
    "table_size_hint": 2003,     # Small prime for laptop
}

# Engram at layers 1 and 3 (early + mid)
ENGRAM_LAYERS = {1, 3}

# Training config
TRAIN_CONFIG = {
    "batch_size": 32,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "num_epochs": 10,
    "grad_clip": 1.0,
    "warmup_steps": 100,
    "log_interval": 50,
}

## 3. Dataset: Character-Level Text

We use a simple character-level dataset. You can replace the text with any file.

In [ ]:
class CharDataset(Dataset):
    """Character-level language modeling dataset."""
    
    def __init__(self, text: str, seq_len: int = 128):
        self.seq_len = seq_len
        # Encode as bytes (0-255)
        self.data = torch.tensor(
            [b for b in text.encode("utf-8", errors="replace")],
            dtype=torch.long
        )
        self.num_samples = max(0, len(self.data) - seq_len - 1)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.seq_len + 1]
        x = chunk[:-1]   # input
        y = chunk[1:]    # target (next character)
        return x, y


# Generate synthetic training data (replace with real text for better results)
# This creates repetitive patterns that the model can learn
SAMPLE_TEXT = """
The quick brown fox jumps over the lazy dog. A stitch in time saves nine.
To be or not to be, that is the question. All that glitters is not gold.
Actions speak louder than words. Knowledge is power. Time is money.
The early bird catches the worm. Practice makes perfect.
Every cloud has a silver lining. Fortune favors the bold.
Where there is a will, there is a way. Rome was not built in a day.
""" * 500  # Repeat to get enough data

dataset = CharDataset(SAMPLE_TEXT, seq_len=CONFIG["max_seq_len"])
dataloader = DataLoader(dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True, drop_last=True)

print(f"Dataset size: {len(dataset):,} samples")
print(f"Batches per epoch: {len(dataloader)}")
print(f"Text length: {len(SAMPLE_TEXT):,} characters")

## 4. Build the Model

In [ ]:
model = CombinedEngramAttnResTransformer(
    vocab_size=CONFIG["vocab_size"],
    hidden_dim=CONFIG["hidden_dim"],
    num_heads=CONFIG["num_heads"],
    num_layers=CONFIG["num_layers"],
    ffn_dim=CONFIG["ffn_dim"],
    block_size=CONFIG["block_size"],
    engram_layers=ENGRAM_LAYERS,
    engram_config=ENGRAM_CONFIG,
    max_seq_len=CONFIG["max_seq_len"],
    dropout=CONFIG["dropout"],
).to(device)

# Parameter summary
summary = model.param_summary()
print("=" * 50)
print("Model Parameter Summary")
print("=" * 50)
print(f"Total parameters:    {summary['total']:>10,}")
print(f"Backbone parameters: {summary['backbone']:>10,}")
print(f"Engram parameters:   {summary['engram']:>10,} ({summary['engram_pct']:.1f}%)")
print(f"AttnRes parameters:  {summary['attn_res']:>10,} ({summary['attn_res_pct']:.1f}%)")
print("=" * 50)

## 5. Training Loop

Following the paper's recommendation:
- AdamW for backbone
- Higher LR for Engram embedding tables (5x multiplier)
- Cosine LR schedule with warmup

In [ ]:
def get_param_groups(model, base_lr, weight_decay):
    """Separate param groups: higher LR for Engram embeddings, no decay for norms/biases."""
    engram_embed_params = []
    no_decay_params = []
    regular_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "embed_tables" in name or "compressor" in name:
            engram_embed_params.append(param)
        elif "norm" in name or "bias" in name or name.endswith(".query"):
            no_decay_params.append(param)
        else:
            regular_params.append(param)
    
    return [
        {"params": regular_params, "lr": base_lr, "weight_decay": weight_decay},
        {"params": no_decay_params, "lr": base_lr, "weight_decay": 0.0},
        {"params": engram_embed_params, "lr": base_lr * 5.0, "weight_decay": 0.0},  # 5x LR, no decay
    ]


def get_lr(step, warmup_steps, max_steps, base_lr):
    """Cosine schedule with linear warmup."""
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


param_groups = get_param_groups(
    model, TRAIN_CONFIG["learning_rate"], TRAIN_CONFIG["weight_decay"]
)
optimizer = torch.optim.AdamW(param_groups)

print(f"Param group sizes: {[sum(p.numel() for p in g['params']) for g in param_groups]}")
print(f"Param group LRs:   {[g['lr'] for g in param_groups]}")

In [ ]:
def train_epoch(model, dataloader, optimizer, epoch, config):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    start_time = time.time()
    
    max_steps = config["num_epochs"] * len(dataloader)
    
    for batch_idx, (input_ids, labels) in enumerate(dataloader):
        global_step = epoch * len(dataloader) + batch_idx
        
        # Update learning rate
        lr = get_lr(global_step, config["warmup_steps"], max_steps, config["learning_rate"])
        for pg in optimizer.param_groups:
            # Scale relative to base LR
            scale = pg["lr"] / config["learning_rate"]
            pg["lr"] = lr * scale
        
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        # Forward
        logits, loss = model(input_ids, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % config["log_interval"] == 0:
            avg_loss = total_loss / num_batches
            elapsed = time.time() - start_time
            tokens_per_sec = num_batches * config["batch_size"] * CONFIG["max_seq_len"] / elapsed
            print(
                f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(dataloader)} | "
                f"Loss: {avg_loss:.4f} | LR: {lr:.2e} | "
                f"{tokens_per_sec:.0f} tok/s"
            )
    
    return total_loss / max(num_batches, 1)

In [ ]:
# Main training loop
print("Starting training...")
print("=" * 60)

losses = []
for epoch in range(TRAIN_CONFIG["num_epochs"]):
    epoch_start = time.time()
    avg_loss = train_epoch(model, dataloader, optimizer, epoch, TRAIN_CONFIG)
    epoch_time = time.time() - epoch_start
    losses.append(avg_loss)
    
    print(f"Epoch {epoch+1}/{TRAIN_CONFIG['num_epochs']} | "
          f"Avg Loss: {avg_loss:.4f} | Time: {epoch_time:.1f}s")
    print("-" * 60)

print("\nTraining complete!")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Loss reduction: {losses[0]:.4f} -> {losses[-1]:.4f} ({(1 - losses[-1]/losses[0])*100:.1f}% decrease)")

## 6. Loss Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(losses) + 1), losses, 'b-o', linewidth=2, markersize=6)
    plt.xlabel("Epoch")
    plt.ylabel("Average Loss")
    plt.title("Training Loss: Engram + AttnRes Combined Model")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available, printing losses instead:")
    for i, l in enumerate(losses):
        bar = '#' * int(50 * l / max(losses))
        print(f"  Epoch {i+1:2d}: {l:.4f} |{bar}")

## 7. Text Generation

In [ ]:
@torch.no_grad()
def generate(model, prompt: str, max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 40):
    """Generate text from a prompt."""
    model.eval()
    
    # Encode prompt as bytes
    tokens = list(prompt.encode("utf-8"))
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        # Truncate to max_seq_len
        if input_ids.size(1) > CONFIG["max_seq_len"]:
            input_ids = input_ids[:, -CONFIG["max_seq_len"]:]
        
        logits, _ = model(input_ids)
        logits = logits[:, -1, :] / temperature  # Last position
        
        # Top-k filtering
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
        
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)
    
    # Decode
    output_bytes = bytes(input_ids[0].tolist())
    return output_bytes.decode("utf-8", errors="replace")


# Generate samples
prompts = ["The quick", "Knowledge is", "Every cloud"]
for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    generated = generate(model, prompt, max_new_tokens=100, temperature=0.7)
    print(f"Generated: {generated}")
    print("-" * 60)

## 8. Inspect Engram Gating Patterns

Visualize which tokens activate the Engram gating mechanism most strongly.

In [ ]:
@torch.no_grad()
def inspect_engram_gates(model, text: str):
    """Extract gating activations from Engram layers."""
    model.eval()
    tokens = list(text.encode("utf-8")[:CONFIG["max_seq_len"]])
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    
    gates = {}
    
    # Hook into engram modules to capture gate values
    hooks = []
    for name, module in model.named_modules():
        if hasattr(module, 'engram') and hasattr(module, 'has_engram') and module.has_engram:
            layer_idx = module.layer_idx
            
            # Capture the engram's gating by running retrieval + partial fusion
            engram_mod = module.engram
            memory = engram_mod.retrieve(input_ids)
            k = engram_mod.W_K(memory)
            
            # We need the hidden state at this layer, but for inspection
            # we'll just use a proxy: the key norm magnitude
            k_norm = engram_mod.key_norm(k)
            gate_magnitude = k_norm.abs().mean(dim=-1)  # [1, L]
            gates[f"layer_{layer_idx}"] = gate_magnitude[0].cpu().numpy()
    
    return gates, tokens


text = "The quick brown fox jumps over the lazy dog."
gates, tokens = inspect_engram_gates(model, text)

print(f"Text: '{text}'")
print()
for layer_name, gate_vals in gates.items():
    print(f"{layer_name} gate magnitudes:")
    chars = [chr(t) if 32 <= t < 127 else '.' for t in tokens]
    for i, (ch, g) in enumerate(zip(chars, gate_vals)):
        bar = '#' * int(g * 20)
        print(f"  [{i:2d}] '{ch}' : {g:.4f} |{bar}")
    print()

## 9. Inspect AttnRes Depth Mixing Weights

Visualize which blocks each layer attends to most.

In [ ]:
@torch.no_grad()
def inspect_attn_res_weights(model):
    """Show learned depth-mixing preferences per layer."""
    model.eval()
    
    print("AttnRes Pseudo-Query Norms (larger = more selective):")
    print("=" * 50)
    
    for i, layer in enumerate(model.layers):
        attn_q_norm = layer.attn_res.query.norm().item()
        mlp_q_norm = layer.mlp_res.query.norm().item()
        
        attn_bar = '#' * int(attn_q_norm * 100)
        mlp_bar = '#' * int(mlp_q_norm * 100)
        
        engram_marker = " [ENGRAM]" if layer.has_engram else ""
        print(f"Layer {i:2d}{engram_marker}")
        print(f"  Attn-side: {attn_q_norm:.4f} |{attn_bar}")
        print(f"  MLP-side:  {mlp_q_norm:.4f} |{mlp_bar}")

inspect_attn_res_weights(model)

## 10. Comparison: Combined vs Baseline vs Individual

Compare model architectures to see the benefit of combining both techniques.

In [ ]:
from attention_residuals import AttnResTransformer


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def quick_train(model, dataloader, num_steps=200, lr=3e-4):
    """Train for a fixed number of steps and return final loss."""
    model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    losses = []
    step = 0
    for input_ids, labels in dataloader:
        if step >= num_steps:
            break
        input_ids, labels = input_ids.to(device), labels.to(device)
        
        logits, loss = model(input_ids, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        losses.append(loss.item())
        step += 1
    
    return losses


# 1. AttnRes-only model
model_attnres = AttnResTransformer(
    vocab_size=256, hidden_dim=128, num_heads=4,
    num_layers=6, ffn_dim=256, block_size=4, max_seq_len=128,
)

# 2. Combined model (Engram + AttnRes)
model_combined = CombinedEngramAttnResTransformer(
    vocab_size=256, hidden_dim=128, num_heads=4,
    num_layers=6, ffn_dim=256, block_size=4,
    engram_layers={1, 3}, engram_config=ENGRAM_CONFIG, max_seq_len=128,
)

print(f"AttnRes-only params:  {count_params(model_attnres):,}")
print(f"Combined params:      {count_params(model_combined):,}")
print()

NUM_COMPARE_STEPS = 300
print(f"Training each for {NUM_COMPARE_STEPS} steps...")

losses_attnres = quick_train(model_attnres, dataloader, NUM_COMPARE_STEPS)
print(f"  AttnRes-only:  initial={losses_attnres[0]:.4f}, final={losses_attnres[-1]:.4f}")

losses_combined = quick_train(model_combined, dataloader, NUM_COMPARE_STEPS)
print(f"  Combined:      initial={losses_combined[0]:.4f}, final={losses_combined[-1]:.4f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Smooth with moving average
    def smooth(vals, window=10):
        smoothed = []
        for i in range(len(vals)):
            start = max(0, i - window)
            smoothed.append(sum(vals[start:i+1]) / (i - start + 1))
        return smoothed
    
    ax.plot(smooth(losses_attnres), label="AttnRes Only", alpha=0.8)
    ax.plot(smooth(losses_combined), label="Engram + AttnRes (Combined)", alpha=0.8)
    
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Loss")
    ax.set_title("Model Comparison: Engram + AttnRes vs AttnRes Only")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available for plotting")

## 11. Save Model

In [ ]:
save_path = "combined_model_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "engram_config": ENGRAM_CONFIG,
    "engram_layers": list(ENGRAM_LAYERS),
    "losses": losses,
}, save_path)
print(f"Model saved to {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB")

---

## Summary

This notebook demonstrates a combined **Engram + Attention Residuals** architecture:

| Component | What it does | Overhead |
|-----------|-------------|----------|
| **Engram** (DeepSeek) | O(1) hash-based memory lookup for factual knowledge injection | Embedding tables (offloadable to DRAM) |
| **AttnRes** (Kimi) | Learned attention over depth - selective access to any prior layer's output | ~1 vector per layer (negligible) |
| **Combined** | Engram injects knowledge; AttnRes routes it selectively across depth | Both overheads, but complementary |

### Key Takeaways:
1. **Engram** excels at factual recall (+12.8 on NIAH in the paper)
2. **AttnRes** excels at reasoning (+7.5 on GPQA-Diamond in the paper)
3. **Combined**, Engram-enriched layers remain accessible to all downstream layers via AttnRes, preventing dilution